In [1]:
from torchvision.datasets import CelebA
from torchvision import transforms

from torchvision.models import ResNet18_Weights

import torchvision
from torch.utils.data import random_split
from torch import nn
from torch.nn import functional as F
from torch.utils.data import TensorDataset, DataLoader

import pytorch_lightning as pl

from torchmetrics.classification import BinaryAUROC, Accuracy
import torch
from pytorch_lightning.loggers import WandbLogger

import numpy as np 

import wandb

from datetime import datetime
from tqdm.notebook import tqdm

In [2]:
DATA_ROOT = "../../datasets"

WANDB_PROJECT = "xaikd-training-teacher-models"
WANDB_GROUP = "celeba"

NUM_WORKERS = 16
BATCH_SIZE = 64
NUM_ATTRIBUTES = 40

NUM_EPOCHS = 20

SEED = 1

DEVICE = "cuda"

ARCH = "resnet18"

# debug
# WANDB_PROJECT = "kitchen-sink"

In [3]:
TRANSFORMATION_DEFAULT = ResNet18_Weights.IMAGENET1K_V1.transforms()
TRANSFORMATION_TRAINING = transforms.Compose(
    [
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=TRANSFORMATION_DEFAULT.mean,
            std=TRANSFORMATION_DEFAULT.std
        )
    ]
)

In [4]:
TRANSFORMATION_DEFAULT

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)

In [5]:
ds_train = CelebA(
    root=DATA_ROOT, split="train", target_type="attr",
    transform=TRANSFORMATION_TRAINING
)

# debug
# trng = torch.Generator()
# trng.manual_seed(1)
# ds_train, _ = random_split(ds_train, [0.1, 0.9], generator=trng)

In [6]:
ds_val = CelebA(
    root=DATA_ROOT, split="valid", target_type="attr",
    transform=TRANSFORMATION_DEFAULT
)

In [7]:
class MultiTaskHead(nn.Module):
    def __init__(self, in_dims, num_tasks, out_per_task):
        super().__init__()
        
        arr_heads = []
        for tix in range(num_tasks):
            head = nn.Linear(in_features=in_dims, out_features=out_per_task)
            arr_heads.append(head)
        self.arr_heads = nn.ModuleList(arr_heads)
        self.task_id = None

    def forward(self, x):
        if self.task_id is not None:
            return self.arr_heads[self.task_id](x)
    
        arr_out = []

        
        for head in self.arr_heads:
            headout = head(x)
            b, d = headout.shape
            arr_out.append(headout.reshape(b, 1, d))

        out = torch.cat(arr_out, dim=1)
        return out

def ano():
    mth = MultiTaskHead(10, 40, 2)
    out = mth(torch.randn(5, 10))
    print(out.shape)
ano()

torch.Size([5, 40, 2])


In [26]:
def get_model(model_name):

    if model_name == "resnet18":
        model = torchvision.models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1, num_classes=1000)
        out_dims, in_dims = model.fc.weight.shape
        for param in model.parameters():
            param.requires_grad = False

        model.fc = nn.Linear(in_features=in_dims, out_features=NUM_ATTRIBUTES)
        
        # model.fc = MultiTaskHead(in_dims, NUM_ATTRIBUTES, 2)
    
    return model

_m = get_model("resnet18")
_m(torch.randn((5, 3, 224, 224)))

tensor([[ 6.8058e-02,  3.4343e-01, -4.0595e-02, -5.7745e-01, -3.6075e-01,
          5.4251e-01,  5.6087e-01,  2.2884e-01, -6.5649e-01, -3.5182e-01,
         -1.2322e-01, -1.4921e-01, -2.8403e-01,  9.2189e-01,  1.2920e-01,
         -2.3661e-01, -1.3184e-01,  3.0889e-01, -2.4531e-01, -2.8656e-01,
         -3.4155e-01, -4.7928e-01, -3.8342e-01, -7.7739e-01, -4.2090e-01,
         -7.3031e-01,  3.5677e-01, -1.1028e-01,  4.5349e-01, -3.7330e-01,
          2.1367e-01, -1.8218e-01, -7.4183e-01, -1.7553e-01,  4.6149e-01,
         -3.5222e-01,  4.3644e-01,  2.5139e-01, -1.2398e+00,  8.7708e-02],
        [-8.0376e-01,  3.6849e-01,  5.8403e-02, -8.5738e-01, -3.7523e-01,
          6.6500e-01,  9.0346e-01, -1.5117e-01, -2.7134e-01,  2.2147e-01,
         -8.0203e-02,  7.4989e-02, -2.5394e-01,  1.2869e-01, -9.1410e-02,
          6.2613e-01, -3.8749e-02,  1.6481e-01, -7.8456e-01, -2.4002e-01,
         -2.2755e-01,  1.4485e-01,  1.4987e-01, -4.4475e-01,  2.4782e-01,
          3.6779e-01,  4.5897e-01, -1

In [27]:
_m.fc.weight.shape

torch.Size([40, 512])

In [28]:
class ModelWrapper(pl.LightningModule):
    def __init__(self, encoder, lr=1e-3):
        super().__init__()

        self.encoder = encoder
        
        self.lr = lr

        self.arr_metrics = dict(train=[], val=[])
        for tix in range(NUM_ATTRIBUTES):
            self.arr_metrics["train"].append(BinaryAUROC(thresholds=20))
            self.arr_metrics["val"].append(BinaryAUROC(thresholds=20))
            # self.arr_metrics["train"].append(Accuracy(task="multiclass", num_classes=2))
            # self.arr_metrics["val"].append(Accuracy(task="multiclass", num_classes=2))

    def forward(self, x):
        embedding = self.encoder(x)
        return embedding

    def configure_optimizers(self):

        optimizer = torch.optim.Adam(
            self.parameters(), lr=self.lr
        )

        return optimizer

    def compute_metric(self, batch, prefix):

        x, y = batch

        arr_logits = self.encoder(x)

        loss = 0
        for tidx in range(NUM_ATTRIBUTES):
            y_task = y[:, tidx]
            logits_task = arr_logits[:, tidx]
            loss += (1/NUM_ATTRIBUTES)*F.binary_cross_entropy_with_logits(
                logits_task, 
                y_task.float()
            )
            self.arr_metrics[prefix][tidx].update(
                logits_task.detach().cpu(),
                y_task.detach().long().cpu()
            )
        
        return loss
    
    def training_step(self, train_batch, batch_idx):

        loss = self.compute_metric(train_batch, "train")

        self.log("train_loss", loss, on_epoch=True, prog_bar=True)

        return loss

        
    def validation_step(self, batch, batch_idx):

        loss = self.compute_metric(batch, "val")

        self.log("val_loss", loss, on_epoch=True, prog_bar=True)
        return loss

    def summary_metrics(self, prefix):
        arr_values = []
        for tix in range(NUM_ATTRIBUTES):
            metric = self.arr_metrics[prefix][tix]
            value = float(metric.compute())
            
            value = 1-value if value < 0.5 else value
            arr_values.append(value)
            self.log(f"{prefix}_auroc_task_{tix}", value)
            metric.reset()
            
        self.log(f"{prefix}_auroc_acc", np.mean(arr_values))

    def on_validation_epoch_end(self):
        self.summary_metrics("val")
    
    def on_train_epoch_end(self):
        self.summary_metrics("train")

In [29]:
model.fc.weight.shape

torch.Size([1000, 512])

In [30]:
pl.seed_everything(SEED)

dl_train = DataLoader(
    ds_train,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    shuffle=True,
)

dl_val = DataLoader(
    ds_val,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,    
    shuffle=False,
)

model  = get_model(ARCH)

wandb_logger = WandbLogger(
    save_dir="/tmp",
    project=WANDB_PROJECT,
    log_model=True,
    group=WANDB_GROUP,
    config=dict(arch=ARCH, seed=SEED, batch_size=BATCH_SIZE, epochs=NUM_EPOCHS)
)

trainer = pl.Trainer(
    max_epochs=NUM_EPOCHS,
    accelerator=DEVICE,
    logger=wandb_logger
)

trainer.fit(
    ModelWrapper(model), 
    dl_train, 
    dl_val,
)
wandb.finish()

[rank: 0] Global seed set to 1


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA A100-PCIE-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type   | Params
-----------------------------------
0 | encoder | ResNet | 11.2 M
-----------------------------------
20.5 K    Trainable params
11.2 M    Non-trainable params
11.2 M    Total params
44.788    Total estimated model params size (MB)


Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

...

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

`Trainer.fit` stopped: `max_epochs=20` reached.


epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇████
train_auroc_acc,▁▇▇█████████████████
train_auroc_task_0,▁▆▇▇███▇▇██▇█▇██████
train_auroc_task_1,▁▇▇▆▆▇▇▇██▇▇▇▇▇▇██▇█
train_auroc_task_10,▁▆█▇█▇████▇██▇█▇▆█▇█
train_auroc_task_11,▁▆▆▇▇▇█▇▇▇██▇▇█▇▇▇██
train_auroc_task_12,▁▆▇▇█▇█▆▇█▇▇███▇▇█▇█
train_auroc_task_13,▁▆▇▇▇▇▇▇▇▇▇▇▇▆▇██▇▇█
train_auroc_task_14,▁▆▆█▇▇▆▇▇▇█▇▇▆▇▇██▆▆
train_auroc_task_15,▁▇▇▇▇▇▇█▇▆▇▇▆▆▇█▇▇▇▆
train_auroc_task_16,▁▆▆▇▇▇▇▇▇█▇▇▇▇▇▇█▇▇▇


In [31]:
model.eval();

In [32]:
def estimate_task_performance(model, dl, task_id):
    metric = BinaryAUROC(thresholds=20)
    # metric = Accuracy(task="multiclass", num_classes=2)
    model.to(DEVICE)

    for x, y in tqdm(dl):
        x = x.to(DEVICE)
        task_logit = model(x)[:, task_id].cpu()
        task_target = y[:, task_id]
        metric.update(task_logit, task_target)

    auroc = float(metric.compute())
    auroc = 1-auroc if auroc < 0.5 else auroc
    return auroc
    
estimate_task_performance(model, dl_val, 0)

  0%|          | 0/311 [00:00<?, ?it/s]

0.891678512096405

In [33]:
estimate_task_performance(model, dl_val, 16)

  0%|          | 0/311 [00:00<?, ?it/s]

0.887769877910614

In [34]:
estimate_task_performance(model, dl_val, 19)

  0%|          | 0/311 [00:00<?, ?it/s]

0.8547618389129639

In [35]:
print(f"Finished at {datetime.now()}")

Finished at 2024-12-18 15:17:34.953696


## 2024/12/16
- [x] check that auroc is equal
- [x] remove debug
- [x] switch to actual project